# 🧠 Emotion Classification Model

This notebook implements a text classification model using a pretrained transformer (XLM-RoBERTa) to predict emotion labels from textual data.

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import torch

# HuggingFace
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

c:\Users\a7mda\OneDrive\Desktop\emotion-physical-classifier\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 📂 Dataset Description

The dataset consists of textual inputs labeled with emotion categories.

- Input: text
- Output: emotion label
- The dataset was cleaned by removing missing values.

We also performed a train/validation/test split to evaluate model performance.

In [3]:
# Load dataset
df = pd.read_csv("../datasets/data.csv")
# Remove missing values
df = df.dropna()

# Display basic info
print(df.head())
print(df["emotion"].value_counts())

                                      text  emotion physical_state
0                  I feel very tired today  sadness          tired
1         Je suis très fatigué aujourd'hui  sadness          tired
2          I am exhausted after a long day  sadness          tired
3  Je suis épuisé après une longue journée  sadness          tired
4    I feel full of energy and ready to go      joy      energetic
emotion
sadness    37
joy        31
calm       20
anger      17
fear       15
Name: count, dtype: int64


## 🧹 Data Preprocessing

The preprocessing steps include:

- Removing missing values
- Encoding labels into numerical format
- Splitting data into training, validation, and test sets

This ensures that the model can learn effectively and be evaluated on unseen data.

In [4]:
# Create label mappings
emotion_labels = sorted(df["emotion"].unique())
emotion2id = {label: i for i, label in enumerate(emotion_labels)}
id2emotion = {i: label for label, i in emotion2id.items()}

# Map labels
df["label"] = df["emotion"].map(emotion2id)

print(emotion2id)

{'anger': 0, 'calm': 1, 'fear': 2, 'joy': 3, 'sadness': 4}


- Splitting data into training, validation, and test sets

In [5]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 84, Val: 18, Test: 18


In [6]:
train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
val_dataset = Dataset.from_pandas(val_df[["text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

## 🔤 Tokenization

We use the XLM-RoBERTa tokenizer to convert text into numerical tokens.

Tokenization includes:
- Padding
- Truncation

This step prepares the text for input into the transformer model.

In [7]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding=True)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 18/18 [00:00<00:00, 5157.28 examples/s]


## 🤖 Model Architecture

We use a pretrained transformer model (XLM-RoBERTa) for sequence classification.

- Backbone: XLM-RoBERTa
- Task: Multi-class classification
- Output: predicted label

The classification head is added on top of the pretrained model and trained on our dataset.

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=len(emotion_labels),
    id2label=id2emotion,
    label2id=emotion2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5673.78it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 📊 Evaluation Metrics

To evaluate the model performance, we define custom metrics using the `compute_metrics` function.

The following metrics are used:

- **Accuracy**: Measures the proportion of correct predictions.
- **F1-score (weighted)**: Provides a balanced measure that accounts for both precision and recall, while handling class imbalance.

These metrics allow us to assess both overall performance and the model’s ability to handle different classes effectively.

In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)
    
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

## ⚙️ Training Configuration

The model is trained using:

- Optimizer: AdamW (via HuggingFace Trainer)
- Learning rate: 2e-5
- Batch size: 8
- Epochs: 15


In [10]:
training_args = TrainingArguments(
    output_dir="./results_emotion",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",                # 🔥 save every epoch
    load_best_model_at_end=True,          # 🔥 reload best model
    metric_for_best_model="f1",           # 🔥 choose metric
    greater_is_better=True,
    logging_steps=10
)

## 📈 Training Results

The model shows progressive improvement during training.

We observe:
- Decreasing training loss
- Increasing accuracy and F1-score

This indicates that the model is learning meaningful patterns from the data.

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

c:\Users\a7mda\OneDrive\Desktop\emotion-physical-classifier\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.063812,1.095049,0.555556,0.415825
2,0.931571,1.015945,0.555556,0.425926
3,0.737574,0.983886,0.611111,0.523232
4,0.769317,0.882094,0.666667,0.596491
5,0.625307,0.868274,0.555556,0.490653
6,0.567056,0.797029,0.611111,0.558606
7,0.501677,0.707536,0.777778,0.768519
8,0.480382,0.799049,0.833333,0.853439
9,0.466302,0.760268,0.777778,0.787302
10,0.385273,0.750420,0.833333,0.829412


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]
c:\Users\a7mda\OneDrive\Desktop\emotion-physical-classifier\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.60s/it]
c:\Users\a7mda\OneDrive\Desktop\emotion-physical-classifier\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.44s/it]
c:\Users\a7mda\OneDrive\Desktop\emotion-physical-classifier\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader

TrainOutput(global_step=165, training_loss=0.49291389855471524, metrics={'train_runtime': 277.4407, 'train_samples_per_second': 4.542, 'train_steps_per_second': 0.595, 'total_flos': 9065242252080.0, 'train_loss': 0.49291389855471524, 'epoch': 15.0})

## 📊 Final Test Results

The final emotion classification model achieved:

- Accuracy: 88.9%
- F1-score: 88.3%

The model demonstrates strong generalization performance, as the test results are consistent with validation performance. This indicates that the model effectively learned meaningful patterns from the dataset.

In [15]:
results = trainer.evaluate(test_dataset)
print(results)

c:\Users\a7mda\OneDrive\Desktop\emotion-physical-classifier\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.175641,0.266002,15,0.888889,0.882716


{'eval_loss': 0.26600244641304016, 'eval_accuracy': 0.8888888888888888, 'eval_f1': 0.882716049382716}


In [16]:
model.save_pretrained("emotion_model")
tokenizer.save_pretrained("emotion_model")

print("✅ Emotion model saved!")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.31s/it]

✅ Emotion model saved!
